# CareerLift AI — Demo Notebook

**Language-Fair Interview Coach built on the AWS Strands Agents SDK**

This notebook demonstrates:
- FairScore: separating **technical correctness** from **communication clarity**
- Plain English and advanced English getting the **same technical band**
- Fairness gap measurement across paraphrase pairs
- Misconception detection and tagging
- All five learning phases working end-to-end

Runs in **mock mode** — no AWS credentials needed.

---
**Live demo:** https://3-235-53-227.nip.io  
**Source:** https://github.com/naveensri4518/CareerLift-AI  
**Hackathon:** Strands SDK: Workshop to Ship, AWS Builder Center 2026

## Setup — clone repo and set mock mode

In [ ]:
import subprocess, sys, os

if not os.path.exists('CareerLift-AI'):
    subprocess.run(['git','clone','https://github.com/naveensri4518/CareerLift-AI.git'], check=True)
os.chdir('CareerLift-AI')
print('Directory:', os.getcwd())

subprocess.run([sys.executable,'-m','pip','install','-q',
    'fastapi','uvicorn','pydantic','strands-agents'], check=True)
print('Dependencies OK')

In [ ]:
import os
os.environ['CAREERLIFT_MODEL_MODE'] = 'mock'
print('Model mode:', os.environ['CAREERLIFT_MODEL_MODE'])
print('Deterministic offline mode — no cloud calls required.')

---
## Part 1 — The Fairness Problem

Do these two answers deserve the same technical score?

| Answer | Register |
|--------|----------|
| *binary search need sorted array, it cut half each time, log n steps* | Plain English |
| *Binary search operates on a pre-sorted array, halving the search space, achieving O(log n)* | Advanced English |

They should. CareerLift AI ensures they do.

In [ ]:
from agent.app import run

BANDS = ['Weak', 'Partial', 'Solid', 'Strong']
CONCEPTS = ['C-BINSEARCH']

plain    = 'binary search need sorted array, it cut half each time, log n steps'
advanced = 'Binary search operates on a pre-sorted array, halving the search space on each iteration, achieving O(log n).'

pr = run({'text': plain,    'expected_concepts': CONCEPTS})['plan']
ar = run({'text': advanced, 'expected_concepts': CONCEPTS})['plan']

print('Plain English    -> technical:', pr['technical_band'], '  comm:', pr['communication_band'])
print('Advanced English -> technical:', ar['technical_band'], '  comm:', ar['communication_band'])

pb = pr['technical_band']; ab = ar['technical_band']
gap = abs(BANDS.index(pb) - BANDS.index(ab)) if pb and ab else 'N/A'
print('Fairness gap:', gap, '(0 = fair)')

---
## Part 2 — Paraphrase Pair Test (5 CS concepts)

Each pair has the same technical content in plain and advanced English.  
Expected fairness gap = 0 for every pair.

In [ ]:
PAIRS = [
    ('C-HASH-AVG',
     'hash map is fast, O(1) most time. if many keys hit same bucket it slow, O(n) worst.',
     'A hash table provides amortized O(1) lookups; pathological collisions degrade it to O(n).'),
    ('C-BINSEARCH',
     'binary search need sorted array, cut half each time, log n steps.',
     'Binary search operates on a pre-sorted array, halving search space, achieving O(log n).'),
    ('C-RECURSION-BASE',
     'function calls itself is recursion. need a stop case or it crash.',
     'Recursion invokes itself; a base case is essential to terminate and avoid stack overflow.'),
    ('C-BIGO-WORSTCASE',
     'worst case not same as average. quicksort usually n log n but n squared with bad pivot.',
     'Quicksort exhibits O(n log n) average yet degrades to O(n^2) under adversarial pivots.'),
    ('C-SQL-JOIN',
     'inner join only keep matching rows. left join keep all left, null where no match.',
     'INNER JOIN returns only matching rows; LEFT JOIN preserves the left relation with NULLs.'),
]

print(f'{"Concept":<22} {"Plain":<12} {"Advanced":<12} {"Gap":<5} {"Fair?"}')
print('-' * 62)
total = 0
for concept, plain, adv in PAIRS:
    pb = run({'text': plain, 'expected_concepts': [concept]})['plan']['technical_band'] or 'Abstain'
    ab = run({'text': adv,   'expected_concepts': [concept]})['plan']['technical_band'] or 'Abstain'
    gap = abs(BANDS.index(pb) - BANDS.index(ab)) if pb in BANDS and ab in BANDS else 'N/A'
    if isinstance(gap, int): total += gap
    mark = 'OK' if gap == 0 else 'FAIL'
    print(f'{concept:<22} {pb:<12} {ab:<12} {str(gap):<5} {mark}')

print(f'\nTotal gap: {total}  Mean: {total/len(PAIRS):.2f}')
print('Result:', 'FAIR (gap=0 for all pairs)' if total == 0 else 'CHECK FAILURES.md')

---
## Part 3 — Misconception Detection

Explicit wrong assertions get a specific misconception tag.  
Vague or absent answers never get tagged.

In [ ]:
CASES = [
    ('Hash always O(1)',  'C-HASH-AVG',       'hash map is always O(1), it is never O(n) no matter what.',          'M-HASH-ALWAYS-O1'),
    ('Unsorted binary search','C-BINSEARCH',  'binary search works on any array, no need to sort first.',            'M-BINSEARCH-UNSORTED'),
    ('JOIN confusion',    'C-SQL-JOIN',        'inner join and left join are the same thing, both combine tables.',   'M-JOIN-INNER-LEFT'),
    ('Correct answer',    'C-HASH-AVG',        'hash table O(1) average but O(n) worst case under heavy collisions.', None),
]

print(f'{"Case":<28} {"Band":<10} {"Tag detected":<25} {"Correct?"}')
print('-' * 75)
for label, concept, answer, expected_tag in CASES:
    p    = run({'text': answer, 'expected_concepts': [concept]})['plan']
    tags = p.get('misconception_tags', [])
    tag  = tags[0] if tags else 'None'
    exp  = expected_tag or 'None'
    ok   = tag == exp
    print(f'{label:<28} {(p["technical_band"] or "Abstain"):<10} {tag:<25} {"OK" if ok else "FAIL"}')

---
## Part 4 — Honest Abstain (PII gate + empty answers)

PII is blocked before any model call. Empty and off-topic answers abstain rather than fabricate a low score.

In [ ]:
ABSTAIN = [
    ('Empty answer',       '',                                               ['C-HASH-AVG']),
    ('PII: phone number',  'my phone 9876543210, hash map O(1)',             ['C-HASH-AVG']),
    ('PII: email',         'email me priya@example.com, binary search sorted',['C-BINSEARCH']),
    ('Off-topic',          'i had lunch and went to the library',            ['C-SQL-JOIN']),
    ('Vague non-answer',   'um it depends could be fast or slow sometimes',  ['C-HASH-AVG']),
]

print(f'{"Case":<28} {"Path":<18} {"Tech abstain"}')
print('-' * 60)
for label, text, concepts in ABSTAIN:
    out  = run({'text': text, 'expected_concepts': concepts})
    path = out['meta'].get('path', 'agent')
    ta   = 'YES (correct)' if out['plan']['abstained_technical'] else f'NO - band={out["plan"]["technical_band"]}'
    print(f'{label:<28} {path:<18} {ta}')

---
## Part 5 — Live API: all five phases

Calls the live production deployment to show all five phases working.

In [ ]:
import urllib.request, json

BASE = 'https://3-235-53-227.nip.io/api'

def api(method, path, body=None, token=None):
    headers = {'Content-Type': 'application/json'}
    if token: headers['Authorization'] = f'Bearer {token}'
    data = json.dumps(body).encode() if body else None
    req  = urllib.request.Request(BASE + path, data=data, headers=headers, method=method)
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())

print('Health:', api('GET', '/health'))

In [ ]:
reg   = api('POST', '/register', {})
TOKEN = reg['token']; SID = reg['student_id']
print('Registered student:', SID[:24], '...')

# Phase 1 Interview Studio
iv   = api('POST', '/interviews', {'topic': 'hashing', 'level': 'beginner'}, TOKEN)
ans  = 'when you store things by a key looking one up is basically instant most of the time'
sc   = api('POST', '/score', {'interview_id': iv['interview_id'], 'question': iv['question'], 'answer': ans, 'concept_token': []}, TOKEN)
print('Phase 1 - Interview Studio')
print('  tech:', sc['technical_band'], '  comm:', sc['communication_band'], '  mock:', sc['mock'])

# Phase 3 Assessment
att = api('POST', '/assessment/attempts', {'category':'dsa','difficulty':'mixed','num_questions':10,'duration_min':10}, TOKEN)
print('Phase 3 - Assessment: attempt_id=%s  questions=%s' % (att['attempt_id'], att['num_questions']))

# Phase 4 System Design
sc4 = api('GET', '/sysdesign/scenarios', token=TOKEN)
print('Phase 4 - System Design scenarios:', [s['id'] for s in sc4['scenarios']])

# Phase 2 Coding Arena
st = api('GET', '/coding/status', token=TOKEN)
pb = api('GET', '/coding/problems', token=TOKEN)
print('Phase 2 - Coding Arena: sandbox=%s  problems=%s' % (st['sandbox_available'], len(pb['problems'])))

# Phase 0 Learning Intelligence
ov  = api('GET', f'/students/{SID}/overview', token=TOKEN)
nxt = api('GET', f'/students/{SID}/next',     token=TOKEN)
print('Phase 0 - Learning Intelligence: interviews=%s  next_topic=%s' % (ov['interviews_completed'], nxt.get('topic', 'pending')))

---
## Summary

| What was shown | Result |
|---|---|
| FairScore: structural separation of tech and comm | Demonstrated |
| Plain English = Advanced English on technical score | Fairness gap = 0 |
| Misconception detection by name | Correct tags on 3 wrong answers |
| Honest abstain on PII and empty inputs | All blocked correctly |
| All five phases running end-to-end | 200 OK on live API |

---
**Live deployment:** https://3-235-53-227.nip.io  
**Source code:** https://github.com/naveensri4518/CareerLift-AI  
**Author:** Naveensri — Strands SDK: Workshop to Ship, AWS Builder Center Hackathon 2026